# Очистка датасета

In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import dask.dataframe as dd
import catboost

In [2]:
df = pd.read_csv("/home/danila/metrolog/backup/expand.csv")

In [3]:
df.head()

,miinstance_passport,miinstance_name,miinstance_type,miinstance_state_condition,miinstance_tech_condition,issue_date,commissioning_date,date_of_debit,fact_date_of_mk,is_fit,plan_date_of_mk,type_of_fact_mk,type_of_planned_mk,prmk
0,1,Трансформатор тока,ТОЛ-10-1,В эксплуатации,Годен,2008-02-03 0:00:00,NaN,NaN,2026-04-08 0:00:00,True,2034-04-08 0:00:00,Поверка,Поверка,96.0
1,2,Трансформатор тока,ТОЛ-10-1,В эксплуатации,Годен,2008-02-03 0:00:00,NaN,NaN,2019-06-17 0:00:00,True,2027-06-17 0:00:00,Поверка,Поверка,96.0
2,3,Трансформатор тока,ТОЛ-10-1,В эксплуатации,Годен,2008-02-03 0:00:00,NaN,NaN,2019-06-17 0:00:00,True,2027-06-17 0:00:00,Поверка,Поверка,96.0
3,4,Трансформатор тока,ТЛМ-6,В эксплуатации,Годен,1973-07-02 0:00:00,NaN,NaN,2026-02-17 0:00:00,True,2030-02-16 0:00:00,Поверка,Поверка,48.0
4,5,Трансформатор тока,ТЛМ-6,В эксплуатации,Годен,1973-07-02 0:00:00,NaN,NaN,2026-02-17 0:00:00,True,2030-02-16 0:00:00,Поверка,Поверка,48.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264503 entries, 0 to 264502
Data columns (total 14 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   miinstance_passport         264503 non-null  int64  
 1   miinstance_name             264503 non-null  object 
 2   miinstance_type             264503 non-null  object 
 3   miinstance_state_condition  262404 non-null  object 
 4   miinstance_tech_condition   261661 non-null  object 
 5   issue_date                  255678 non-null  object 
 6   commissioning_date          26881 non-null   object 
 7   date_of_debit               6544 non-null    object 
 8   fact_date_of_mk             232913 non-null  object 
 9   is_fit                      232913 non-null  object 
 10  plan_date_of_mk             228243 non-null  object 
 11  type_of_fact_mk             232934 non-null  object 
 12  type_of_planned_mk          245943 non-null  object 
 13  prmk          

In [5]:
print(df.isnull().sum())

miinstance_passport                0
miinstance_name                    0
miinstance_type                    0
miinstance_state_condition      2099
miinstance_tech_condition       2842
issue_date                      8825
commissioning_date            237622
date_of_debit                 257959
fact_date_of_mk                31590
is_fit                         31590
plan_date_of_mk                36260
type_of_fact_mk                31569
type_of_planned_mk             18560
prmk                           18518
dtype: int64


In [6]:
today = pd.Timestamp.now()

# 1. Возраст
df['issue_date_clean'] = pd.to_datetime(df['issue_date'], errors='coerce')
df['age_years'] = (today - df['issue_date_clean']).dt.days / 365.25

# 2. Дней с последней поверки
df['fact_date_clean'] = pd.to_datetime(df['fact_date_of_mk'], errors='coerce')
df['days_since_last_mk'] = (today - df['fact_date_clean']).dt.days

# 3. Дней до плановой поверки
df['plan_date_clean'] = pd.to_datetime(df['plan_date_of_mk'], errors='coerce')
df['days_until_planned_mk'] = (df['plan_date_clean'] - today).dt.days

# 4. Истёк ли МПИ (исправлено: месяцы → дни)
df['mpi_expired'] = (
    (today - df['fact_date_clean']).dt.days > (df['prmk'] * 30.44)
).astype(int)

# 5. Списание
df['is_written_off'] = df['date_of_debit'].notna().astype(int)

In [7]:
df.head()

,miinstance_passport,miinstance_name,miinstance_type,miinstance_state_condition,miinstance_tech_condition,issue_date,commissioning_date,date_of_debit,fact_date_of_mk,is_fit,...,type_of_planned_mk,prmk,issue_date_clean,age_years,fact_date_clean,days_since_last_mk,plan_date_clean,days_until_planned_mk,mpi_expired,is_written_off
0,1,Трансформатор тока,ТОЛ-10-1,В эксплуатации,Годен,2008-02-03 0:00:00,NaN,NaN,2026-04-08 0:00:00,True,...,Поверка,96.0,2008-02-03,18.357290,2026-04-08,66.0,2034-04-08,2855.0,0,0
1,2,Трансформатор тока,ТОЛ-10-1,В эксплуатации,Годен,2008-02-03 0:00:00,NaN,NaN,2019-06-17 0:00:00,True,...,Поверка,96.0,2008-02-03,18.357290,2019-06-17,2553.0,2027-06-17,368.0,0,0
2,3,Трансформатор тока,ТОЛ-10-1,В эксплуатации,Годен,2008-02-03 0:00:00,NaN,NaN,2019-06-17 0:00:00,True,...,Поверка,96.0,2008-02-03,18.357290,2019-06-17,2553.0,2027-06-17,368.0,0,0
3,4,Трансформатор тока,ТЛМ-6,В эксплуатации,Годен,1973-07-02 0:00:00,NaN,NaN,2026-02-17 0:00:00,True,...,Поверка,48.0,1973-07-02,52.947296,2026-02-17,116.0,2030-02-16,1343.0,0,0
4,5,Трансформатор тока,ТЛМ-6,В эксплуатации,Годен,1973-07-02 0:00:00,NaN,NaN,2026-02-17 0:00:00,True,...,Поверка,48.0,1973-07-02,52.947296,2026-02-17,116.0,2030-02-16,1343.0,0,0


In [8]:
## Выводим кол-во значений типа null в процентах
print(df.isnull().sum() / len(df) * 100)

miinstance_passport            0.000000
miinstance_name                0.000000
miinstance_type                0.000000
miinstance_state_condition     0.793564
miinstance_tech_condition      1.074468
issue_date                     3.336446
commissioning_date            89.837166
date_of_debit                 97.525926
fact_date_of_mk               11.943154
is_fit                        11.943154
plan_date_of_mk               13.708729
type_of_fact_mk               11.935214
type_of_planned_mk             7.016934
prmk                           7.001055
issue_date_clean               3.336824
age_years                      3.336824
fact_date_clean               11.943154
days_since_last_mk            11.943154
plan_date_clean               13.708729
days_until_planned_mk         13.708729
mpi_expired                    0.000000
is_written_off                 0.000000
dtype: float64


## Разметка данных

Т.к. данные взяты с боевой базы данных, их нужно разметить, размечать будем основываясь не на отдельных значениях, а системно анализируя целый ряд значений
Основываться будет на:
 - Штатном состоянии (miinstance_state_condition)
 - Техническом состоянии (miinstance_tech_condition)
 - бинарном флаге mpi_expired
 - И флаг "Годен/Не годен" (is_fit)
 - Дата списания (date_of_debit)
 - Наличие плановой поверки (plan_date_of_mk)

In [9]:
# Создаём свежую копию
df_labeled = df.copy()

# 1. Все → consider
df_labeled['label'] = 'consider'

# 2. not_buy (перезапишет consider, но будет перезаписан buy)
not_buy_mask = (
    (df_labeled['miinstance_state_condition'] == 'В эксплуатации') &
    (df_labeled['miinstance_tech_condition'].isin(['Годен', 'Исправен'])) &
    (df_labeled['is_fit'] == True)
)
df_labeled.loc[not_buy_mask, 'label'] = 'not_buy'

# 3. buy (перезаписывает и consider, и not_buy)
buy_states = ['На списание', 'Списан', 'Удален из базы ВХ', 'Утерян']
df_labeled.loc[df_labeled['miinstance_state_condition'].isin(buy_states), 'label'] = 'buy'

buy_tech = ['Не годен', 'Просрочен', 'В отказе метрол.', 'В отказе явном']
df_labeled.loc[df_labeled['miinstance_tech_condition'].isin(buy_tech), 'label'] = 'buy'

df_labeled.loc[df_labeled['date_of_debit'].notna(), 'label'] = 'buy'

# 4. Результат
print("=== ИТОГОВОЕ РАСПРЕДЕЛЕНИЕ ===")
print(df_labeled['label'].value_counts())
print(df_labeled['label'].value_counts(normalize=True))

# 5. Проверка: сколько потенциальных not_buy стали not_buy
not_buy_potential = (
    (df['miinstance_state_condition'] == 'В эксплуатации') &
    (df['miinstance_tech_condition'].isin(['Годен', 'Исправен'])) &
    (df['is_fit'] == True)
)

final_not_buy = df_labeled.loc[not_buy_potential, 'label']
print(f"\nИз {not_buy_potential.sum()} потенциальных not_buy:")
print(final_not_buy.value_counts())

=== ИТОГОВОЕ РАСПРЕДЕЛЕНИЕ ===
label
not_buy     116256
buy         115473
consider     32774
Name: count, dtype: int64
label
not_buy     0.439526
buy         0.436566
consider    0.123908
Name: proportion, dtype: float64

Из 119697 потенциальных not_buy:
label
not_buy    116256
buy          3441
Name: count, dtype: int64


In [10]:
df_labeled.info

<bound method DataFrame.info of         miinstance_passport            miinstance_name miinstance_type  \
0                         1         Трансформатор тока        ТОЛ-10-1   
1                         2         Трансформатор тока        ТОЛ-10-1   
2                         3         Трансформатор тока        ТОЛ-10-1   
3                         4         Трансформатор тока           ТЛМ-6   
4                         5         Трансформатор тока           ТЛМ-6   
...                     ...                        ...             ...   
264498               299361  Термометр биметаллический             БТ5   
264499               299362                  Микрометр           МК 25   
264500               299363                  Микрометр           МК 25   
264501               299364                  Микрометр           МК 50   
264502               299365                  Микрометр           МК 50   

       miinstance_state_condition miinstance_tech_condition  \
0               

## Обработка значений от nan

In [11]:
## обработка числовых столбцов
numeric_cols = df_labeled.select_dtypes(include=['int64', 'float64']).columns
df_labeled[numeric_cols] = df_labeled[numeric_cols].fillna(df_labeled[numeric_cols].mean())

objects_cols = df_labeled.select_dtypes(include=['object']).columns
df_labeled[objects_cols] = df_labeled[objects_cols].astype(str).replace('nan', 'unknown')
df_labeled.isnull().sum()

miinstance_passport               0
miinstance_name                   0
miinstance_type                   0
miinstance_state_condition        0
miinstance_tech_condition         0
issue_date                        0
commissioning_date                0
date_of_debit                     0
fact_date_of_mk                   0
is_fit                            0
plan_date_of_mk                   0
type_of_fact_mk                   0
type_of_planned_mk                0
prmk                              0
issue_date_clean               8826
age_years                         0
fact_date_clean               31590
days_since_last_mk                0
plan_date_clean               36260
days_until_planned_mk             0
mpi_expired                       0
is_written_off                    0
label                             0
dtype: int64

## Обучение модели

Обучение модели будем производить методом категориального бустинга
Почему?
Потому что это метод ансамблевого обучения, который:
 - Отлично работает с табличными данными
 - Потребляет меньше оперативной памяти
 - Дает достаточно хорошую точность результатов
 - Удобство работы с категориальными признаками
 - Эффективно работает с разреженными данными и пропусками

In [12]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

categorial_features = [
    'miinstance_state_condition',
    'miinstance_tech_condition',
    'type_of_fact_mk',
    'type_of_planned_mk',
]

# Числовые признаки (которые мы подготовили)
numeric_features = [
    'age_years',
    'days_since_last_mk',
    'days_until_planned_mk',
    'mpi_expired',
    'is_written_off'
]

features = categorial_features + numeric_features

X = df_labeled[features].copy()
y = df_labeled['label'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

model = CatBoostClassifier (
    auto_class_weights = 'Balanced',
    iterations = 500,
    learning_rate = 0.1,
    depth = 6,
    random_seed = 42,
    verbose = 100,
    eval_metric = 'MultiClass'
)

model.fit(
    X_train, y_train,
    cat_features = categorial_features,
    eval_set = (X_test, y_test),
    verbose = 100,
    plot = False
)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


0:	learn: 0.9460150	test: 0.9464742	best: 0.9464742 (0)	total: 203ms	remaining: 1m 41s
100:	learn: 0.0094416	test: 0.0115314	best: 0.0115314 (100)	total: 6.17s	remaining: 24.4s
200:	learn: 0.0068972	test: 0.0091515	best: 0.0091515 (200)	total: 13.7s	remaining: 20.3s
300:	learn: 0.0058328	test: 0.0084014	best: 0.0084014 (300)	total: 21.9s	remaining: 14.5s
400:	learn: 0.0051625	test: 0.0079955	best: 0.0079862 (394)	total: 30.2s	remaining: 7.45s
499:	learn: 0.0046908	test: 0.0078627	best: 0.0078555 (482)	total: 38.9s	remaining: 0us

bestTest = 0.007855536831
bestIteration = 482

Shrink model to first 483 iterations.
              precision    recall  f1-score   support

         buy       1.00      1.00      1.00     23095
    consider       1.00      0.99      1.00      6555
     not_buy       1.00      1.00      1.00     23251

    accuracy                           1.00     52901
   macro avg       1.00      1.00      1.00     52901
weighted avg       1.00      1.00      1.00     52901

In [18]:
# Создаем тестовый пример с двумя разными вариантами
test_multiple = pd.DataFrame({
    'miinstance_state_condition': ['В эксплуатации', 'Списан'],
    'miinstance_tech_condition': ['Годен', 'Не годен'],
    'is_fit': [True, True],
    'age_years': [20.0, 20.0],
    'days_since_last_mk': [10000, 10000],
    'days_until_planned_mk': [400, 400],
    'mpi_expired': [1, 1],
    'is_written_off': [0, 1],
    'type_of_fact_mk': ['периодическая', 'периодическая'],
    'type_of_planned_mk': ['плановая', 'плановая']
})

# Обработка категориальных признаков
for col in categorial_features:
    if col in test_multiple.columns:
        test_multiple[col] = test_multiple[col].astype(str).replace('nan', 'unknown')

# Предсказание для всех строк сразу
predictions = model.predict(test_multiple[features])
probabilities = model.predict_proba(test_multiple[features])

print(f"Предсказания: {predictions}")
print(f"\nВероятности (формат: {len(predictions)} строк × {len(model.classes_)} классов):")
print(probabilities)

Предсказания: [['not_buy']
 ['buy']]

Вероятности (формат: 2 строк × 3 классов):
[[0.00101407 0.00512307 0.99386286]
 [0.9922423  0.00354339 0.00421431]]


In [14]:
# model.save_model('buyer_model.cbm')

# import joblib

# joblib.dump(features, 'model_files/model_features.pkl')
# joblib.dump(categorial_features, 'model_files/categorial_features.pkl')
# joblib.dump(model.classes_, 'model_files/model_classes.pkl')


In [15]:
# import sys
# print("Python executable:", sys.executable)
# print("Python version:", sys.version)
# print("Module search paths:", sys.path)